# 02.01 - Add the release year to the song tables

**Goal:** give every song in the song tables a `release_year` (the year the music came out), so we can say something about **era**: how old the music in a genre is, and how old the music is when people listen to it.

The song tables (`songs_genre_YEAR_dev.parquet`) already have a `release_mbid`: an id that names one release (an album or a single). It is just a code, it does not say *when*. MusicBrainz knows when, so we look it up. **The linking and the genres are not touched.**

The way to the year:

```
release_mbid  ->  MusicBrainz table `release`  ->  its release group  ->  `release_group_meta`: first_release_date_year
```

A *release group* is "the album" as a whole, with all its editions (CD, vinyl, reissue). Its `first_release_date_year` is the year of the earliest edition.

Steps:
1. Collect the different `release_mbid` of all years.
2. Look up their release year in MusicBrainz and save it as a small lookup table.
3. Add the column `release_year` to each song table (one file at a time, written to a temporary file first, then put in place of the old one).
4. Check that nothing else changed, and look at how many listens got a year.

You can run this notebook again: if a table already has `release_year`, it is replaced by the new one.

**A limit to remember:** if the `release_mbid` is a compilation or a reissue, the year is the year of *that* release, so an old song can look newer than it is.

## 1. Setup

In [ ]:
%pip install -q duckdb pyarrow

In [ ]:
from pathlib import Path
import os, time
import duckdb
import pandas as pd

RAW = Path("/Users/hanshornbek/Desktop/Projects/Data Mining project")
MB = RAW / "Raw data/MusicBrainz Full/tables/mbdump"                       # the MusicBrainz database tables

DEV_ONLY = os.environ.get("DEV_ONLY", "1") == "1"
SUFFIX = "dev" if DEV_ONLY else "all"
YEARS = range(2006, 2027)
MEMORY_LIMIT = "3GB"

OUT = Path.cwd().parent / "Output_dev"                                      # the output folder (change here if it gets another name)
SONGS = OUT / "data" / "parquets" / "songs"                                 # the song tables from notebook 02
LOOKUPS = OUT / "data" / "parquets" / "lookups"                             # the small lookup table is saved here
RY = OUT / "release_year"                                                   # the coverage table is saved here
LOOKUPS.mkdir(parents=True, exist_ok=True)
RY.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

def run(sql):
    con.execute(sql).fetchall()

def scalar(sql):
    return con.execute(sql).fetchall()[0][0]

run(f"PRAGMA memory_limit='{MEMORY_LIMIT}'")
run(f"PRAGMA temp_directory='{OUT / 'data' / '.tmp'}'")

def tsv(name, columns):
    '''SQL for reading one MusicBrainz table (tab-separated, no header, a backslash-N marks a missing value).'''
    cols = ", ".join(f"'{c}': 'VARCHAR'" for c in columns)
    return (f"read_csv('{MB / name}', delim='\\t', header=false, quote='', escape='', nullstr='\\N', "
            f"columns={{{cols}}}, auto_detect=false, strict_mode=false, ignore_errors=true)")

songs_found = sorted(SONGS.glob(f"songs_genre_*_{SUFFIX}.parquet"))
print(f"{len(songs_found)} song tables found in {SONGS}")

## 2. Which releases do we need?

All the different `release_mbid` in the song tables. Songs without one cannot get a year.

In [ ]:
files = SONGS / f"songs_genre_*_{SUFFIX}.parquet"
run(f'''CREATE OR REPLACE TABLE wanted AS
        SELECT DISTINCT release_mbid FROM read_parquet('{files}')
        WHERE release_mbid IS NOT NULL AND release_mbid <> '' ''')
print(f"{scalar('SELECT count(*) FROM wanted'):,} different releases in the song tables")

## 3. Look up the release year in MusicBrainz

Two tables, both tab-separated text files:

- `release`: one row per release. We need its id (`gid`, the same as our `release_mbid`) and which release group it belongs to.
- `release_group_meta`: one row per release group, with `first_release_date_year`.

The `release` table is 800 MB; DuckDB reads it directly and keeps only the rows of our releases. This takes a minute or two.

In [ ]:
start = time.time()
RELEASE_COLUMNS = ["id", "gid", "name", "artist_credit", "release_group", "status", "packaging", "language",
                   "script", "barcode", "comment", "edits_pending", "quality", "last_updated"]
META_COLUMNS = ["id", "release_count", "first_release_date_year", "first_release_date_month",
                "first_release_date_day", "rating", "rating_count"]

run(f'''CREATE OR REPLACE TABLE rel AS
        SELECT gid AS release_mbid, release_group::BIGINT AS release_group
        FROM {tsv('release', RELEASE_COLUMNS)}
        WHERE gid IN (SELECT release_mbid FROM wanted)''')
print(f"{scalar('SELECT count(*) FROM rel'):,} of the releases found in MusicBrainz ({time.time() - start:.0f} s)")

run(f'''CREATE OR REPLACE TABLE lookup_raw AS
        SELECT r.release_mbid, m.first_release_date_year::INT AS year_raw
        FROM rel r
        LEFT JOIN {tsv('release_group_meta', META_COLUMNS)} m ON m.id::BIGINT = r.release_group''')
print(f"{scalar('SELECT count(*) FROM lookup_raw WHERE year_raw IS NOT NULL'):,} of them have a year ({time.time() - start:.0f} s)")

Years that cannot be right (before 1900 or after 2026) are set to missing, so they do not disturb the averages. How many are there?

In [ ]:
print("years before 1900:", scalar("SELECT count(*) FROM lookup_raw WHERE year_raw < 1900"))
print("years after 2026: ", scalar("SELECT count(*) FROM lookup_raw WHERE year_raw > 2026"))

run('''CREATE OR REPLACE TABLE lookup AS
       SELECT release_mbid, CASE WHEN year_raw BETWEEN 1900 AND 2026 THEN year_raw END AS release_year
       FROM lookup_raw''')

# CHECK: one row per release, so the join in the next step cannot make a song appear twice
assert scalar("SELECT count(*) - count(DISTINCT release_mbid) FROM lookup") == 0, "a release_mbid appears twice in the lookup"

run(f"COPY lookup TO '{LOOKUPS / 'release_year_lookup.parquet'}' (FORMAT parquet)")
print(f"{scalar('SELECT count(*) FROM lookup WHERE release_year IS NOT NULL'):,} releases with a usable year. Saved: release_year_lookup.parquet")
con.sql("SELECT release_year, count(*) AS releases FROM lookup WHERE release_year IS NOT NULL GROUP BY 1 ORDER BY 1 DESC LIMIT 8").df()

## 4. Add the column to the song tables

For each year:
1. read the song table and join the lookup on `release_mbid` (a `LEFT JOIN`, so songs without a year stay in, with an empty year),
2. write the result to a temporary file (`.part`),
3. check that the number of songs and the listens are exactly the same as before,
4. only then put the new file in place of the old one.

If something fails in step 1-3, the old file is not touched. A leftover `.part` file can be deleted.

In [ ]:
coverage = []
for year in YEARS:
    start = time.time()
    f = SONGS / f"songs_genre_{year}_{SUFFIX}.parquet"
    tmp = f.with_name(f.name + ".part")
    columns = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{f}')").fetchall()]
    drop = "EXCLUDE (release_year)" if "release_year" in columns else ""          # a re-run replaces the old column

    before = con.execute(f"SELECT count(*), sum(n_listens) FROM read_parquet('{f}')").fetchall()[0]
    run(f'''COPY (SELECT s.* {drop}, l.release_year
                  FROM read_parquet('{f}') s
                  LEFT JOIN lookup l ON l.release_mbid = s.release_mbid) TO '{tmp}' (FORMAT parquet)''')
    after = con.execute(f"SELECT count(*), sum(n_listens) FROM read_parquet('{tmp}')").fetchall()[0]
    assert before == after, f"{year}: songs or listens changed ({before} -> {after}), the old file is kept"

    row = con.execute(f'''SELECT {year} AS year,
                                 sum(n_listens) AS listens,
                                 coalesce(sum(n_listens) FILTER (WHERE release_mbid IS NOT NULL AND release_mbid <> ''), 0) AS listens_with_release_id,
                                 coalesce(sum(n_listens) FILTER (WHERE release_year IS NOT NULL), 0) AS listens_with_year,
                                 sum(n_listens * ({year} - release_year)) FILTER (WHERE release_year IS NOT NULL)
                                   / sum(n_listens) FILTER (WHERE release_year IS NOT NULL) AS avg_song_age
                          FROM read_parquet('{tmp}')''').df()          # from the new file, before it replaces the old one
    os.replace(tmp, f)
    coverage.append(row)
    print(f"{year}: {after[0]:>9,} songs, {int(row.loc[0, 'listens_with_year']):>10,} of {int(row.loc[0, 'listens']):>10,} listens have a year ({time.time() - start:.0f} s)")

## 5. How many listens have a release year?

`pct_with_release_id` is the share of listens whose song has a release id, `pct_with_year` the share that also has a year. `avg_song_age` is the average age of the music in the year it was played (listening year minus release year), over the listens that have a year.

In [ ]:
coverage = pd.concat(coverage, ignore_index=True)
for c in ["listens_with_release_id", "listens_with_year"]:
    coverage["pct_" + c.replace("listens_with_", "with_")] = (100 * coverage[c] / coverage["listens"]).round(1)
coverage["avg_song_age"] = coverage["avg_song_age"].round(1)
coverage.to_csv(RY / f"release_year_coverage_{SUFFIX}.csv", index=False)
print("saved", RY / f"release_year_coverage_{SUFFIX}.csv")
coverage[["year", "listens", "pct_with_release_id", "pct_with_year", "avg_song_age"]]